In [16]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-huggingface langchain-text-splitters sentence-transformers chromadb ipywidgets

In [17]:
# Store the biological baseline information in a Python string

baseline_text = """
BIOLOGICAL BASELINES FOR LAB MICE

Heart Rate:
Normal range: 500 to 700 beats per minute.
Any value below 500 bpm or above 700 bpm should be flagged as an anomaly.

Body Temperature:
Normal range: 36.5 to 38.0 degrees Celsius.
Any value below 36.5 C or above 38.0 C should be flagged as an anomaly.

Respiratory Rate:
Normal range: 80 to 200 breaths per minute.
Any value below 80 breaths per minute or above 200 breaths per minute should be flagged as an anomaly.

ANALYSIS RULES:

1. Extract the subject ID and test case ID if available.
2. Extract all available biological measurements.
3. Compare each measurement with the corresponding normal baseline.
4. Mark each measurement as NORMAL or ANOMALY.
5. Clearly explain why a measurement is abnormal.
6. Do not invent missing measurements.
7. Only use the biological baseline information provided in this document.
"""

# Create a text file named baselines.txt
# This file will later be used by the RAG system

with open("baselines.txt", "w") as f:
    f.write(baseline_text)

print("✅ Biological baseline document created.")

✅ Biological baseline document created.


In [18]:
# Import the Colab userdata module
# It allows us to securely access secrets stored in Google Colab

from google.colab import userdata

# Import the os module to store the API key as an environment variable

import os

# Get the OpenRouter API key from Colab Secrets

OPENROUTER_API_KEY = userdata.get("GenAi_Chatbot")

# Check whether the API key was found

if not OPENROUTER_API_KEY:
    raise ValueError("❌ OpenRouter API key not found. Check Colab Secrets.")

# Store the API key as an environment variable

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print("✅ OpenRouter API key loaded successfully.")

✅ OpenRouter API key loaded successfully.


In [19]:
# Import the OpenRouter-compatible LangChain LLM

from langchain_openai import ChatOpenAI

# Import the text document loader

from langchain_community.document_loaders import TextLoader

# Import the text splitter used to divide documents into smaller chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import ChromaDB vector store

from langchain_chroma import Chroma

# Import HuggingFace embeddings

from langchain_huggingface import HuggingFaceEmbeddings

# Import widgets for creating the input box and button

import ipywidgets as widgets

# Import display functions for showing output in Colab

from IPython.display import display, Markdown, clear_output

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [20]:
# Load the biological baseline text file

loader = TextLoader("baselines.txt")

# Read the document

documents = loader.load()

# Create a text splitter
# The document will be divided into smaller pieces for RAG

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split the document into chunks

splits = splitter.split_documents(documents)

# Create the embedding model
# It converts text into numerical vectors

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store the document chunks inside ChromaDB

vector_store = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

# Create a retriever
# k=3 means the system retrieves the 3 most relevant chunks

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("✅ RAG system created successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ RAG system created successfully.


In [21]:
# Ask the RAG system for the biological normal ranges

results = retriever.invoke(
    "What are the normal heart rate, temperature and respiratory rate ranges for lab mice?"
)

# Print the retrieved information

print("===== BIOLOGICAL BASELINE =====")

for doc in results:
    print(doc.page_content)

===== BIOLOGICAL BASELINE =====
BIOLOGICAL BASELINES FOR LAB MICE

Heart Rate:
Normal range: 500 to 700 beats per minute.
Any value below 500 bpm or above 700 bpm should be flagged as an anomaly.

Body Temperature:
Normal range: 36.5 to 38.0 degrees Celsius.
Any value below 36.5 C or above 38.0 C should be flagged as an anomaly.

Respiratory Rate:
Normal range: 80 to 200 breaths per minute.
Any value below 80 breaths per minute or above 200 breaths per minute should be flagged as an anomaly.

ANALYSIS RULES:
BIOLOGICAL BASELINES FOR LAB MICE

Heart Rate:
Normal range: 500 to 700 beats per minute.
Any value below 500 bpm or above 700 bpm should be flagged as an anomaly.

Body Temperature:
Normal range: 36.5 to 38.0 degrees Celsius.
Any value below 36.5 C or above 38.0 C should be flagged as an anomaly.

Respiratory Rate:
Normal range: 80 to 200 breaths per minute.
Any value below 80 breaths per minute or above 200 breaths per minute should be flagged as an anomaly.

ANALYSIS RULES:
ANAL

In [22]:
# Create the Large Language Model using OpenRouter

llm = ChatOpenAI(
    # OpenRouter free model routing
    model="openrouter/free",

    # Low temperature gives more consistent results
    temperature=0.1,

    # Pass the API key
    api_key=OPENROUTER_API_KEY,

    # OpenRouter API endpoint
    base_url="https://openrouter.ai/api/v1"
)

print("✅ OpenRouter LLM initialized.")

✅ OpenRouter LLM initialized.


In [23]:
# Send a simple test message to the LLM

test_response = llm.invoke(
    "Reply with exactly: Bio-Sync AI is working."
)

# Display the response

print(test_response.content)

Bio-Sync AI is working.


In [24]:
# Define the Ingestor Agent as a Python function

def ingestor_agent(test_case):

    # Create the prompt for the Ingestor Agent

    prompt = f"""
You are the Bio-Sync Ingestor Agent.

Read the laboratory test case below.

Extract ONLY information that is actually present.

Extract:

- Test Case ID
- Subject ID
- Heart Rate
- Body Temperature
- Respiratory Rate

Return the result in this exact format:

Test Case ID:
Subject ID:
Heart Rate:
Body Temperature:
Respiratory Rate:

If something is missing, write:
Not Provided

Do NOT guess or invent missing values.

LABORATORY TEST CASE:
{test_case}
"""

    # Send the prompt to the LLM

    response = llm.invoke(prompt)

    # Return the extracted information

    return response.content

In [25]:
# Define the Analyst Agent

def analyst_agent(extracted_data):

    # Query used to search the biological baseline document

    query = """
What are the normal biological baseline ranges for:
heart rate,
body temperature,
and respiratory rate
in laboratory mice?
"""

    # Search the ChromaDB vector store

    baseline_results = retriever.invoke(query)

    # Combine the retrieved document chunks

    baseline_information = "\n\n".join(
        doc.page_content for doc in baseline_results
    )

    # Create the Analyst Agent prompt

    prompt = f"""
You are the Bio-Sync Analyst Agent.

You must analyze laboratory measurements using ONLY
the biological baseline information provided below.

BIOLOGICAL BASELINE FROM RAG:
{baseline_information}

EXTRACTED TEST DATA:
{extracted_data}

Compare every measurement that is actually provided.

Use this exact structure:

Heart Rate:
Actual Value:
Normal Range:
Status:
Reason:

Body Temperature:
Actual Value:
Normal Range:
Status:
Reason:

Respiratory Rate:
Actual Value:
Normal Range:
Status:
Reason:

Overall Result:

Rules:

1. Use NORMAL when the value is inside the normal range.
2. Use ANOMALY when the value is outside the normal range.
3. Do not analyze missing measurements.
4. Do not invent measurements.
5. Clearly explain abnormal measurements.
6. Use only the biological baseline provided by RAG.
7. Do not give a medical diagnosis.
"""

    # Send the analysis request to the LLM

    response = llm.invoke(prompt)

    # Return the analysis

    return response.content

In [26]:
# Define the Summarizer Agent

def summarizer_agent(analysis):

    # Create the report-generation prompt

    prompt = f"""
You are the Bio-Sync Summarizer Agent.

Create a professional laboratory test case report
from the Analyst Agent's findings.

ANALYST FINDINGS:
{analysis}

Use this format:

# BIO-SYNC TEST CASE REPORT

## Test Case Information

Test Case ID:
Subject ID:

## Overall Result

## Measurement Analysis

### 1. Heart Rate

Actual Value:
Normal Range:
Status:
Reason:

### 2. Body Temperature

Actual Value:
Normal Range:
Status:
Reason:

### 3. Respiratory Rate

Actual Value:
Normal Range:
Status:
Reason:

## Abnormal Measurements

## Normal Measurements

## Executive Summary

## Recommended Follow-up

Rules:

1. Only use information provided by the Analyst Agent.
2. Do not invent missing information.
3. Do not provide a medical diagnosis.
4. If a measurement was not provided, write "Not Provided".
5. Keep the report clear and professional.
"""

    # Send the prompt to the LLM

    response = llm.invoke(prompt)

    # Return the final report

    return response.content

In [27]:
# Define the complete Bio-Sync pipeline

def biosync_pipeline(test_case):

    # ==========================================
    # AGENT 1: INGESTOR
    # ==========================================

    print("🔵 Ingestor Agent running...")

    # Send the user's test case to the Ingestor Agent

    extracted_data = ingestor_agent(test_case)

    print("✅ Ingestor Agent completed.")
    print()

    # ==========================================
    # AGENT 2: ANALYST
    # ==========================================

    print("🟢 Analyst Agent running...")

    # The Analyst will use RAG to retrieve the
    # biological baseline information

    print("🔎 Searching biological baseline using RAG...")

    # Analyze the extracted measurements

    analysis = analyst_agent(extracted_data)

    print("✅ Analyst Agent completed.")
    print()

    # ==========================================
    # AGENT 3: SUMMARIZER
    # ==========================================

    print("🟣 Summarizer Agent running...")

    # Convert the Analyst findings into the final report

    final_report = summarizer_agent(analysis)

    print("✅ Summarizer Agent completed.")
    print()

    # Return the final report

    return final_report

In [28]:
# Create a sample laboratory test case

test_case = """
TC004 Mouse 004: HR 600 bpm, Temp 37.0 C, Resp 120 bpm.
"""

# Run the complete Bio-Sync pipeline

report = biosync_pipeline(test_case)

# Display the generated report

print("========================================")
print("FINAL REPORT")
print("========================================")

print(report)

🔵 Ingestor Agent running...
✅ Ingestor Agent completed.

🟢 Analyst Agent running...
🔎 Searching biological baseline using RAG...
✅ Analyst Agent completed.

🟣 Summarizer Agent running...
✅ Summarizer Agent completed.

FINAL REPORT
# BIO-SYNC TEST CASE REPORT

## Test Case Information

Test Case ID: Not Provided  
Subject ID: Not Provided  

## Overall Result  
NORMAL  

## Measurement Analysis  

### 1. Heart Rate  
Actual Value: 600 bpm  
Normal Range: 500 to 700 beats per minute  
Status: NORMAL  
Reason: The actual heart rate of 600 bpm falls within the specified normal range of 500 to 700 beats per minute.  

### 2. Body Temperature  
Actual Value: 37.0 C  
Normal Range: 36.5 to 38.0 degrees Celsius  
Status: NORMAL  
Reason: The recorded body temperature of 37.0 C is within the specified normal range of 36.5 to 38.0 degrees Celsius.  

### 3. Respiratory Rate  
Actual Value: 120 bpm  
Normal Range: 80 to 200 breaths per minute  
Status: NORMAL  
Reason: The observed respiratory ra

In [29]:
# Create a large text box for entering the laboratory test case

test_case_box = widgets.Textarea(
    value="",

    # Placeholder shown when the box is empty

    placeholder="""Enter your laboratory test case here...

Example:
TC001 Mouse 001 has heart rate 750 bpm,
temperature 37.2 C and respiratory rate 90 bpm.""",

    # Label displayed beside the textbox

    description="Test Case:",

    # Set the size of the textbox

    layout=widgets.Layout(
        width="100%",
        height="180px"
    )
)

# Create the Generate Report button

generate_button = widgets.Button(
    description="Generate Report",

    # Green button

    button_style="success",

    # Set button size

    layout=widgets.Layout(
        width="220px",
        height="45px"
    )
)

# Create an output area
# The final report will appear here

output = widgets.Output()

# Display the textbox

display(test_case_box)

# Display the button

display(generate_button)

# Display the output area

display(output)

Textarea(value='', description='Test Case:', layout=Layout(height='180px', width='100%'), placeholder='Enter y…

Button(button_style='success', description='Generate Report', layout=Layout(height='45px', width='220px'), sty…

Output()

In [30]:
# Define what should happen when the button is clicked

def generate_report(button):

    # Send all output to the output widget

    with output:

        # Clear the previous report

        clear_output()

        # Get the text entered by the user

        test_case = test_case_box.value.strip()

        # Check whether the textbox is empty

        if test_case == "":
            print("❌ Please enter a test case.")
            return

        # Display processing message

        print("⏳ Generating Bio-Sync report...")
        print()

        # Show the agent workflow

        print(
            "🔵 Ingestor Agent → "
            "🟢 Analyst Agent → "
            "🟣 Summarizer Agent"
        )

        print()

        try:

            # Run the complete multi-agent pipeline

            final_report = biosync_pipeline(test_case)

            # Clear the processing messages

            clear_output()

            # Display the final report as Markdown

            display(
                Markdown(final_report)
            )

        except Exception as e:

            # If an error occurs, display the error

            clear_output()

            print("❌ REPORT GENERATION FAILED")
            print()

            print("Error Type:")
            print(type(e).__name__)

            print()

            print("Error:")
            print(str(e))


# Connect the button with the generate_report function

generate_button.on_click(generate_report)

print("✅ Generate Report button is ready.")

✅ Generate Report button is ready.
